In [ ]:
! pip install langchain langchain-openai langchain-community langgraph python-dotenv faiss-cpu pypdf


  Using cached tiktoken-0.14.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (6.7 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 575.5/575.5 kB 24.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 35.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 33.1 MB/s  0:00:00
Using cached tiktoken-0.14.0-cp313-cp313-macosx_11_0_arm64.whl (1.0 MB)
  Attempting uninstall: langchain-core╺━━━━━━━━━━━━━━ 5/8 [openai]
    Found existing installation: langchain-core 1.6.12━━━━━━━━ 5/8 [openai]
    Uninstalling langchain-core-1.6.12:━━━━━━━━━━━━━━ 5/8 [openai]
      Successfully uninstalled langchain-core-1.6.12━━━━━━━━━━ 5/8 [openai]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [langchain-openai][langchain-core]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip


In [14]:
%pip install -U python-dotenv pypdf langchain-community langchain-text-splitters langchain-core langgraph langchain-google-genai langchain-huggingface faiss-cpu sentence-transformers

  Using cached scipy-1.18.1-cp313-cp313-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached joblib-1.6.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached rich-15.0.0-py3-none-any.whl.metadata (18 kB)
  Using cached annotated_doc-0.0.5-py3-none-any.whl.metadata (6.5 kB)
  Using cached markdown_it_py-4.2.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 14.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 20.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 24.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━

In [38]:
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI , GoogleGenerativeAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings 



In [39]:
loader = PyPDFLoader('intro-to-ml.pdf')
docs = loader.load()

In [40]:
len(docs)

392

In [41]:
splliter = RecursiveCharacterTextSplitter(chunk_size = 1000 , chunk_overlap = 200)
chunks = splliter.split_documents(docs)

In [42]:
len(chunks)

973

In [43]:
embeddings = HuggingFaceEmbeddings(model_name ="google/embeddinggemma-300m",encode_kwargs={
        "truncate_dim": 128,
        "normalize_embeddings": True
    })
vector_store = FAISS.from_documents(chunks , embeddings)

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 6327.22it/s]


In [44]:
vector_store

In [45]:
retriever = vector_store.as_retriever(search_type='similarity',serach_kwags={'k':4}) 

In [46]:
response = retriever.invoke('What is decision tree')

print(len(response))

4


In [47]:
def rag_tool(query):
    """
    Retriver relevant information from the pdf document
    use this tool when the user asks futural / conceptual questions
    that might be answered from the stored documents
    """
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query':query,
        'context': context,
        'metadata':metadata
    }

In [48]:
retriever.invoke('What is decision tree')

[Document(id='b176ff11-19e2-41a6-a506-c7e6da7db8a6', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 'title': 'Introduction to Machine Learning with Python', 'trapped': '/False', 'moddate': '2020-08-19T07:09:16+02:00', 'source': 'intro-to-ml.pdf', 'total_pages': 392, 'page': 0, 'page_label': 'Cover'}, page_content='Andreas C. Müller & Sarah Guido\nIntroduction to \nMachine \nLearning  \nwith P y thon   \nA GUIDE FOR DATA SCIENTISTS'),
 Document(id='057ab179-4040-4ede-97e8-7e8d335fff6c', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 5.9.1.5 (http://www.pdf-tools.com)', 'creator': 'AH CSS Formatter V6.2 MR4 for Linux64 : 6.2.6.18551 (2014/09/24 15:00JST)', 'creationdate': '2016-09-21T13:04:39+00:00', 'author': 'Andreas C. Müller and Sarah Guido', 

In [49]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)
tools = [rag_tool]
tool_model = model.bind_tools(tools)

In [50]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage] , add_messages]

In [51]:
def chat_node(state: ChatState):
    messages = state['messages']

    response = tool_model.invoke(messages)

    return {'messages':[response]}

In [52]:
tool_node = ToolNode(tools)

In [53]:
graph= StateGraph(ChatState)

graph.add_node('chat_node',chat_node)
graph.add_node('tools',tool_node)

graph.add_edge(START , 'chat_node')
graph.add_conditional_edges('chat_node' , tools_condition)
graph.add_edge('tools' ,'chat_node' )

chat_bot  =graph.compile()

In [58]:
prompt = {
    "messages": [
        HumanMessage(
            content=(
                "Using the pdf notes, explain how to find the ideal value of K in KNN"
            )
        )
    ]
}
response = chat_bot.invoke(prompt)

print(response['messages'][-1].text)

GoogleRateLimitError: Error calling model 'gemini-3.6-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.6-flash, description: Generate content requests per day limit exceeded on Free Tier. See https://ai.google.dev/gemini-api/docs/rate-limits.\nPlease retry in 18h26m1.212566218s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'description': '